# 실습 - 문서 질의응답

**문서 이미지에 질문하여 답을 추출**

**작업 개요**

- 작업 이름 — `document-question-answering`
- 입력 — **문서 이미지 + 질문** (두 가지)
- 출력 — 답변 (`answer`)
- 문서의 글자와 배치를 함께 인식
- 문서에 없는 정보는 답할 수 없음 — **추출 방식**

**앞선 작업과의 차이**

| 구분 | 입력 | 출력 |
|---|---|---|
| 분류 · 탐지 · 분할 | 이미지 | 라벨 · 상자 · 마스크 |
| 깊이 추정 | 이미지 | 깊이 값 |
| 설명 생성 | 이미지 | 문장 (자동 생성) |
| **문서 질의응답** | **이미지 + 질문** | **질문에 대한 답** |

**데이터**

- 재무 보고서의 표·도표를 촬영·스캔한 이미지
- `caption` 열 — 문서 내용을 정리한 참고 텍스트
- 문서에 어떤 항목이 있는지 미리 확인 가능

**모델 — Donut (Document Understanding Transformer)**

- OCR 도구 없이 이미지에서 직접 판독
- 별도 도구 설치 불필요

**실습 순서**

- 환경 준비
- 데이터 불러오기 · 문서 내용 확인
- 모델 준비 · 질문 실행
- 답변 검증 · 질문 바꿔보기

---
## 0. 환경 준비

- `datasets` — Hub 데이터셋 내려받기
- `transformers` — 사전학습 모델로 추론 수행

In [ ]:
from datasets import load_dataset
from transformers import pipeline
import torch

print("준비 완료")
print("GPU 사용 가능:", torch.cuda.is_available())

---
## 1. 데이터 준비

### 1-1. 데이터셋 불러오기

- 재무 보고서의 표·도표 이미지
- `split="train[:50]"` — 앞 50장만 사용

In [ ]:
ds = load_dataset(
    "Anas989898/Vision-OCR-Financial-Reports-10k",
    split="train[:50]"
)

print("열:", ds.column_names)
ds

### 1-2. 문서 이미지 확인

- 표와 항목이 배치된 재무 문서
- 이미지 형식이 **RGBA** — 모델 입력을 위해 RGB로 변환

In [ ]:
doc = ds[0]["image"]

print("원본 형식:", doc.mode, doc.size)

doc = doc.convert("RGB")
print("변환 후:", doc.mode)

doc

### 1-3. 문서 내용 확인

- `caption` 열 — 문서 내용을 정리한 참고 텍스트
- 항목별 정답이 아닌 **서술 형태**
- 어떤 항목을 물어볼지 정하는 데 활용

In [ ]:
caption = ds[0]["caption"]

print(caption[:1500])

---
## 2. 모델 준비

- `task="document-question-answering"` — 문서 질의응답 작업
- `model` — Donut (OCR 없이 이미지에서 직접 판독)
- `device=0` — GPU에서 실행

첫 실행 시 모델 내려받기로 시간이 소요된다.

In [ ]:
qa = pipeline(
    task="document-question-answering",
    model="naver-clova-ix/donut-base-finetuned-docvqa",
    device=0
)

print("모델 준비 완료")

---
## 3. 질문 실행

- `image` · `question` — 두 인자를 모두 지정
- 질문은 **영어**로 작성
- 출력은 목록 형태 — `answer` 항목에 답이 담김

In [ ]:
result = qa(image=doc, question="What is the title of this table?")

print(result)
print()
print("답변:", result[0]["answer"])

### 3-1. 여러 질문 실행

- 같은 문서 · 다른 질문 → 다른 답변
- 위에서 확인한 `caption` 내용을 참고하여 질문 구성
- 답변이 맞는지는 **문서 이미지를 보고 직접 확인**

In [ ]:
questions = [
    "What is the title of this table?",
    "What years are covered in this table?",
    "What is the total amount?",
    "What is the unit of the numbers?",
]

for q in questions:
    ans = qa(image=doc, question=q)[0]["answer"]
    print(f"{q}")
    print(f"  → {ans}")
    print()

### 3-2. 문서를 다시 보며 검증

- 답변이 문서에 실제로 있는 값인지 확인
- 답변에 **점수가 없음** — 확신 정도를 알 수 없으므로 직접 검증 필요

In [ ]:
doc

### 3-3. 문서에 없는 정보 질문

- 문서에 없는 내용을 물으면 **엉뚱한 답**을 산출
- 모델은 항상 답을 내놓음 — 정확하다는 뜻은 아님
- **추출 방식** — 문서에 있는 내용만 답할 수 있음

In [ ]:
for q in [
    "What is the weather today?",
    "Who is the author of this document?",
]:
    print(q)
    print("  →", qa(image=doc, question=q)[0]["answer"])
    print()

### 3-4. 질문 표현 바꾸기

- 같은 내용을 다르게 물으면 답변이 달라질 수 있음
- 질문 구성이 결과에 영향 — 제로샷의 후보 지정과 같은 원리

In [ ]:
for q in [
    "What is the total?",
    "What is the total amount?",
    "What is the total value in millions?",
]:
    print(f"{q}")
    print(f"  → {qa(image=doc, question=q)[0]['answer']}")
    print()

---
## 4. 여러 문서 처리

- 문서마다 항목과 배치가 다름
- 동일한 질문으로 여러 문서를 처리 — 자동 정보 추출

In [ ]:
for n in range(3):
    d = ds[n]["image"].convert("RGB")
    ans = qa(image=d, question="What is the title of this table?")[0]["answer"]

    print(f"[{n}] {ans}")
    display(d.resize((400, 300)))
    print()

### 4-1. 표 형태로 정리

- 여러 문서에서 같은 항목을 추출하여 표로 정리
- 문서 더미를 자동으로 데이터화하는 작업에 해당

In [ ]:
import pandas as pd

rows = []
for n in range(10):
    d = ds[n]["image"].convert("RGB")
    rows.append({
        "번호": n,
        "제목": qa(image=d, question="What is the title of this table?")[0]["answer"],
        "기간": qa(image=d, question="What years are covered?")[0]["answer"],
    })

df = pd.DataFrame(rows)
df

**활용**

- 결과를 파일로 저장하여 이후 분석에 사용
- `df.to_csv("result.csv", index=False, encoding="utf-8-sig")`

---
## 5. 다른 문서 확인

- 문서 번호를 바꾸어 다른 보고서 확인
- `caption` 을 먼저 읽고 질문을 구성
- 본인이 보유한 문서 이미지에 적용 — 계약서 · 성적표 · 청구서 등

In [ ]:
n = 5
d = ds[n]["image"].convert("RGB")

print(ds[n]["caption"][:600])
print()
print("답변:", qa(image=d, question="What is the title of this table?")[0]["answer"])

d

---
## 6. 정리

**수행 내용**

- 문서 이미지에 질문하여 항목별 정보 추출
- 질문을 바꾸어 같은 문서에서 다른 정보 획득
- 여러 문서를 동일 질문으로 처리하여 표로 정리

**앞선 작업과의 차이**

- 입력이 **두 가지** — 이미지와 질문
- 사용자가 무엇을 얻을지 **직접 지정**
- 질문 표현이 결과에 영향

**특징과 한계**

- OCR 도구 없이 이미지에서 직접 판독 — 별도 설치 불필요
- 답변에 점수가 없음 — 확신 정도를 알 수 없어 **직접 검증 필요**
- **추출 방식** — 문서에 없는 정보는 답할 수 없음
- 문서에 없는 내용을 물어도 답을 산출

**주의**

- 이미지 형식이 RGBA인 경우 RGB로 변환 후 입력
- 표가 복잡하거나 해상도가 낮으면 정확도 저하

**활용**

- 문서 더미의 자동 데이터화 — 계약서 · 청구서 · 보고서
- 특정 항목만 골라 추출 — 질문으로 지정

---

데이터 출처: Anas989898/Vision-OCR-Financial-Reports-10k (Hugging Face Hub 공개)